<a href="https://colab.research.google.com/github/srivastavask/nlp_work/blob/main/spelling_correction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Bigram Spelling Correction
import urllib.request
from collections import defaultdict
from IPython.display import HTML, display

In [ ]:
print("Downloading dictionary dataset...")
url = "https://raw.githubusercontent.com/dwyl/english-words/master/words_alpha.txt"
response = urllib.request.urlopen(url)
vocabulary = [line.decode('utf-8').strip().lower() for line in response.readlines() if len(line.strip()) > 1]
print(f"Loaded {len(vocabulary):,} words into memory.")

Loaded 370,079 words into memory.


In [ ]:
class BigramInvertedIndex:
    def __init__(self, vocab: list[str]):
        self.vocabulary = vocab
        self.index = defaultdict(set)
        self.word_bigrams = []
        self._build_index()

    def _extract_bigrams(self, word: str) -> set[str]:
        padded = f"${word.lower()}$"
        return {padded[i:i+2] for i in range(len(padded) - 1)}

    def _build_index(self):
        print("Building Inverted Index...")
        for word_id, word in enumerate(self.vocabulary):
            bigrams = self._extract_bigrams(word)
            self.word_bigrams.append(bigrams)
            for bigram in bigrams:
                self.index[bigram].add(word_id)
        print("Index building complete!")

    def search(self, query: str, top_n: int = 5):
        query_bigrams = self._extract_bigrams(query)
        if not query_bigrams:
            return [], 0

        # Retrieve candidates matching shared bigrams
        candidate_counts = defaultdict(int)
        for bigram in query_bigrams:
            for word_id in self.index.get(bigram, []):
                candidate_counts[word_id] += 1

        results = []
        len_q = len(query_bigrams)

        # Calculate exact Jaccard score for candidate pool
        for word_id, shared_count in candidate_counts.items():
            cand_bigrams = self.word_bigrams[word_id]
            union_len = len_q + len(cand_bigrams) - shared_count
            jaccard = shared_count / union_len if union_len > 0 else 0.0

            results.append({
                "word": self.vocabulary[word_id],
                "score": round(jaccard, 4),
                "shared": shared_count
            })

        results.sort(key=lambda x: x["score"], reverse=True)
        return results[:top_n], len(candidate_counts)


# Build the index
engine = BigramInvertedIndex(vocabulary)

Building Inverted Index...
Index building complete!


In [ ]:
def render_ui():
    html_code = """
    <div style="font-family: Arial, sans-serif; max-width: 650px; padding: 15px; border: 1px solid #e0e0e0; border-radius: 8px;">
        <h2>🔤 Character Bigram Spell Checker</h2>
        <p>Type a misspelled word below to search across <b>100,000+ words</b> instantly:</p>
        <input type="text" id="userInput" value="appli" style="padding: 8px; width: 250px; font-size: 16px; border-radius: 4px; border: 1px solid #ccc;"/>
        <button onclick="runSearch()" style="padding: 8px 15px; font-size: 16px; background-color: #0288d1; color: white; border: none; border-radius: 4px; cursor: pointer;">Search</button>
        <div id="results" style="margin-top: 15px;"></div>
    </div>

    <script>
    async function runSearch() {
        let inputVal = document.getElementById('userInput').value;
        let resultDiv = document.getElementById('results');
        resultDiv.innerHTML = "<i>Searching index...</i>";

        let result = await google.colab.kernel.invokeFunction('notebook_search', [inputVal], {});
        let data = result.data['text/html'];
        resultDiv.innerHTML = data;
    }
    runSearch();
    </script>
    """
    display(HTML(html_code))

from google.colab import output

def python_search_callback(query_word):
    results, candidate_count = engine.search(query_word, top_n=5)

    html = f"<p><b>Candidates Evaluated:</b> {candidate_count:,} / {len(vocabulary):,} total words</p>"
    html += "<table border='1' cellpadding='6' cellspacing='0' style='border-collapse: collapse; width: 100%; text-align: left;'>"
    html += "<tr style='background-color: #f2f2f2;'><th>Rank</th><th>Candidate Word</th><th>Jaccard Similarity Score</th><th>Shared Bigrams</th></tr>"

    for i, res in enumerate(results):
        html += f"<tr><td>{i+1}</td><td><b>{res['word']}</b></td><td>{res['score']}</td><td>{res['shared']}</td></tr>"
    html += "</table>"

    return HTML(html)

# Register Colab Python bridge
output.register_callback('notebook_search', python_search_callback)

# Launch UI
render_ui()

In [ ]:
# 1. Install required packages
!pip install -q gradio nltk pyspellchecker

import nltk
import gradio as gr
from nltk.metrics.distance import edit_distance
from spellchecker import SpellChecker

In [ ]:
# Edit distance

# Download NLTK resources
nltk.download('words')
nltk.download('punct')

# Initialize spell checker
spell = SpellChecker()

def process_text(input_word, target_word):
    # Calculate NLTK Edit Distance
    distance = edit_distance(input_word, target_word)

    # Get spelling suggestions using pyspellchecker
    corrected = spell.correction(input_word)
    candidates = spell.candidates(input_word)

    # Format candidates list
    candidates_list = ", ".join(candidates) if candidates else "None"

    # Generate DP Table Visualization for the demo
    m, n = len(input_word), len(target_word)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): dp[i][0] = i
    for j in range(n + 1): dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if input_word[i - 1] == target_word[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])

    matrix_str = f"DP Table ({input_word} -> {target_word}):\n"
    matrix_str += "\t" + "\t".join([""] + list(target_word)) + "\n"
    for i, row in enumerate(dp):
        char = input_word[i - 1] if i > 0 else ""
        matrix_str += f"{char}\t" + "\t".join(map(str, row)) + "\n"

    return distance, corrected, candidates_list, matrix_str

# Build Gradio Web Interface
interface = gr.Interface(
    fn=process_text,
    inputs=[
        gr.Textbox(lines=1, placeholder="Enter input/misspelled word...", label="Input Word"),
        gr.Textbox(lines=1, placeholder="Enter target word to compare...", label="Target Word (for Distance Matrix)")
    ],
    outputs=[
        gr.Number(label="NLTK Edit Distance"),
        gr.Textbox(label="Auto-Corrected Word"),
        gr.Textbox(label="Top Candidates"),
        gr.Textbox(label="Dynamic Programming Matrix", lines=10)
    ],
    title="Edit Distance & Spelling Correction Demo",
    description="Interactive web tool using NLTK and PySpellChecker for string comparison and text correction."
)

# Launch with share link for Google Colab
interface.launch(share=True, debug=True)

[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Unzipping corpora/words.zip.
[nltk_data] Error loading punct: Package 'punct' not found in index


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4edffa145c0b9709c0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
